In [ ]:
import os
import sys
import torch
import tiktoken


device = "cuda" if torch.cuda.is_available() else "cpu"
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"Using device: {device}")

Using device: mps


In [ ]:
config = TransformerConfig(

    context_window=256,

    vocabulary_size=50257,

    num_layers=4,

    num_heads=4,

    embedding_dim=256,

    dropout_rate=0.1
)

model = NanoTransformer(config)

model.to(device)

checkpoint = torch.load(
    "nano_transformer_checkpoint.pt",
    map_location=device
)

model.load_state_dict(
    checkpoint["network_state"]
)

model.eval()

print("Model checkpoint loaded successfully!")

number of parameters: 7.23M
Loaded model from /Users/idant/Developer/Projects/NanoGPT/training/nanogpt_checkpoint_4.pt


In [ ]:
# Setup tokenizer
enc = tiktoken.get_encoding("gpt2")

prompt = "[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]\nI ain't a killer but don't push me"
print(f"\nPrompt: '{prompt}'\n")

idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
max_new_tokens = 500


Prompt: '[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]
I ain't a killer but don't push me'



In [ ]:
print("Temperature Sampling")
# Setting manual seed for determinism in generation examples
torch.manual_seed(42)
out_idx = model.generate(idx, max_new_tokens, temperature=0.7, repetition_penalty=1.2)
print(enc.decode(out_idx[0].tolist()))

Temperature Sampling
[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]
I ain't a killer but don't push me, yeah (Yeah)
They tryna make my style and you know I'ma let it rain (Yeah!)
I swear I was a man that we spent his momma as fuck like "Jatthean'"
For the homie, you're sittin', catch your face in off
And all the fuckin' niggas out of my bitches sayin' a nigga
Put me on, call him up, hit him up, watch him out
You was playin' off the trap up when he got with a tutor
He said she be a dirty, tell her quick for so
She was fucked 'em meanin' to have a motherfucker
Sorry they wishin' too many problems that?
Really just had a few lotta goin' off, bitch
Always wanted to gettin' ass in a baby
No one-money-got nigga say how this shit came
Or maybe I got the same rap game, it's all the same
So what it make me big dick
We gon' lose it, never leave it up
Now it over the street, no way you gotta let me down
It's nothin' gonna see alive
A lot of these hoes and the way

In [ ]:
print("Top-k Sampling")
torch.manual_seed(42)
out_idx = model.generate(idx, max_new_tokens, top_k=15, repetition_penalty=1.2)
print(enc.decode(out_idx[0].tolist()))

Top-k Sampling
[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]
I ain't a killer but don't push me in this rap, nigga that I'm a party
Yeah, nigga, yeah
You know you got my niggas to get your man
Bitches like me and I'm tryna give him
Ayy, we gon' make him see you with us (Ah)
They tell them girls he gon' have a big bitch, they doin', they say it (Oh)
We both on me, I can just need it up
And all the bad shit I feelin' these bitches
It's for them hoes, we bein' up (Hey!)
Now we don't wanna hear how they hate 'em
You'rea have a motherfucker from me, you know what I'm sayin'?
If I keep a little fucking girl, she gotta playin' me (Sayin'-लाए)
I've been a new friend that old friends
Don't leave out there
Just put her down, "Come back," and take your head in the road, go and turn his back"

When they runnin' up when they gettin' the same time we got a lot, no one day
You can take away from the best, we could get your life and your way, baby, hey
All that's a l

In [ ]:
prompt_text = "[Genre: boom_bap] [Mood: introspective] [Rhyme: alternate] [Cadence: midtempo]\nLife is"

input_ids = torch.tensor(
    [TOKENIZER.encode(prompt_text)],
    device=device
)

strategies = {

    "High Temperature": {
        "temperature": 1.5
    },

    "Top-K (k=15)": {
        "temperature": 0.8,
        "top_k": 15
    },

    "Top-P (p=0.90)": {
        "temperature": 0.8,
        "top_p": 0.90
    },

    "Penalized Loops": {
        "temperature": 0.7,
        "top_k": 40,
        "repetition_penalty": 1.5
    }
}

print("--- DECODING STRATEGY COMPARISON ---\n")

for name, cfg in strategies.items():

    out = model.generate(
        input_ids.clone(),
        max_tokens=40,
        **cfg
    )

    decoded = TOKENIZER.decode(
        out[0].tolist()
    )

    print(
        f"[{name} Output]:\n"
        f"{decoded}\n"
        f"{'-'*30}"
    )

In [ ]:
# Stress Test 1: Context Window Degradation (Feed a prompt longer than your max context window)
long_prompt = "word " * 1200  # Exceeds standard 1024 window
input_long = torch.tensor([TOKENIZER.encode(long_prompt[:1024])], device=device) 
print("Testing Context Window limit...")
# Check if model starts hallucinating, looping, or crashing

# Stress Test 2: Infinite Loop Trigger (Low temperature, no penalty)
loop_prompt = "[Genre: trap] [Mood: confident]\nI get money "
input_loop = torch.tensor([TOKENIZER.encode(loop_prompt)], device=device)
loop_out = model.generate(input_loop, max_tokens=50, temperature=0.2, repetition_penalty=1.0)
print("Loop Test Output:\n", TOKENIZER.decode(loop_out[0].tolist()))